# 06_train_quantile_gbm
Task 9. Trains quantile gradient-boosted models (LightGBM) that predict trip minutes for weekday 4-7 PM trips: a p10, p50 and p90 model for each mode (bike, taxi, fhv), 9 models in all.

Same setup as the baseline in 05: train on the months before October, test on October, and a p10-p90 band (80% coverage is the target).

Features are things the app knows when someone asks for a trip: pickup and dropoff zone and borough, hour, day of week, the zone centroids and the distance between them, and traffic volume (zone level, borough average where a zone has no counter). Trip distance and fare are left out on purpose.

Reads from `workspace.cs483`: `trips_zoned`, `taxi_zone_lookup`, `traffic_zone_hourly`, `traffic_borough_hourly`, and `baseline_zone` / `baseline_borough` from 05.

Writes: `taxi_zone_centroids`, `gbm_metrics`, `gbm_test_predictions`, and the models plus the lookups the app needs in the `models` folder of the `nyc_inputs` volume.

In [ ]:
%pip install "lightgbm>=4.7" shapely pyshp

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from pyspark.sql import functions as F

S = "workspace.default"
HOURS = [16, 17, 18]                  # weekday 4-7 PM, same window as the baseline
TEST_MONTH = 10                       # train on months before October, test on October
MODES = ["bike", "taxi", "fhv"]
QUANTILES = [0.1, 0.5, 0.9]           # p10 / p50 / p90, same band as the baseline
LOG_TARGET = True                     # fit log(1 + minutes); quantiles carry through the transform
MAX_TRAIN_ROWS = 1_500_000            # a mode with more training trips than this gets sampled
MODEL_DIR = "/Volumes/workspace/default/nyc_inputs/models"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"

PARAMS = dict(objective="quantile", learning_rate=0.05, num_leaves=63, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              n_estimators=2000, n_jobs=-1, random_state=42, verbose=-1)

## Zone and traffic features

In [ ]:
def zone_centroids():
    """Centroid of every taxi zone in feet (EPSG:2263), from the same shapefile as 03_map_to_zones."""
    import io, tempfile, zipfile
    import requests, shapefile, shapely
    from shapely.geometry import shape
    tmp = tempfile.mkdtemp()
    zipfile.ZipFile(io.BytesIO(requests.get(ZONES_URL, timeout=120).content)).extractall(tmp)
    sf = shapefile.Reader(f"{tmp}/taxi_zones/taxi_zones.shp")
    fields = [f[0] for f in sf.fields[1:]]
    ids = [int(dict(zip(fields, r))["LocationID"]) for r in sf.records()]
    geoms = [shapely.make_valid(shape(s.__geo_interface__)) for s in sf.shapes()]
    rows = []
    for z in sorted(set(ids)):          # some zones are made of several polygons
        c = shapely.union_all([g for i, g in zip(ids, geoms) if i == z]).centroid
        rows.append((z, c.x, c.y))
    return pd.DataFrame(rows, columns=["zone", "x_ft", "y_ft"])


CENT = f"{S}.taxi_zone_centroids"
if spark.catalog.tableExists(CENT):
    cent = spark.table(CENT).toPandas()
else:
    cent = zone_centroids()
    spark.createDataFrame(cent).write.mode("overwrite").saveAsTable(CENT)

lk = spark.table(f"{S}.taxi_zone_lookup").select("zone", "borough").toPandas()
zones = lk.merge(cent, on="zone", how="inner")
zones = zones[zones.zone.between(1, 263)].reset_index(drop=True)
print(len(zones), "zones with a borough and centroid")

In [ ]:
def weekday_hourly(table, key):
    return (spark.table(f"{S}.{table}").filter("is_weekday").filter(F.col("hour").isin(HOURS))
            .select(key, "hour", F.col("avg_vol_per_hour").alias("vol")).toPandas())

tz = weekday_hourly("traffic_zone_hourly", "zone")
tb = weekday_hourly("traffic_borough_hourly", "borough").rename(columns={"vol": "b_vol"})
traffic = (zones[["zone", "borough"]].merge(pd.DataFrame({"hour": HOURS}), how="cross")
           .merge(tz, on=["zone", "hour"], how="left").merge(tb, on=["borough", "hour"], how="left"))
traffic["traffic"] = traffic["vol"].fillna(traffic["b_vol"])      # zone value, else borough average
traffic = traffic[["zone", "hour", "traffic"]]
print(f"{traffic.traffic.notna().mean():.0%} of zone-hours have a traffic number")

## Features, data loading and training

In [ ]:
ZONE_CATS = sorted(int(z) for z in zones.zone)
BORO_CATS = sorted(zones.borough.unique())
CAT_COLS = ["pu_zone", "do_zone", "pu_b", "do_b"]
FEATURES = ["pu_zone", "do_zone", "pu_b", "do_b", "hour", "dow", "pu_x", "pu_y", "do_x", "do_y",
            "dist_mi", "same_zone", "pu_traffic", "do_traffic"]


def build_features(df):
    """Trips (pu_zone, do_zone, hour, dow) -> feature table. Use this same function when predicting."""
    pu = zones.rename(columns={"zone": "pu_zone", "borough": "pu_b", "x_ft": "pu_x", "y_ft": "pu_y"})
    do = zones.rename(columns={"zone": "do_zone", "borough": "do_b", "x_ft": "do_x", "y_ft": "do_y"})
    tpu = traffic.rename(columns={"zone": "pu_zone", "traffic": "pu_traffic"})
    tdo = traffic.rename(columns={"zone": "do_zone", "traffic": "do_traffic"})
    X = (df[["pu_zone", "do_zone", "hour", "dow"]].reset_index(drop=True)
         .merge(pu, on="pu_zone", how="left").merge(do, on="do_zone", how="left")
         .merge(tpu, on=["pu_zone", "hour"], how="left").merge(tdo, on=["do_zone", "hour"], how="left"))
    X["dist_mi"] = np.hypot(X.pu_x - X.do_x, X.pu_y - X.do_y) / 5280
    X["same_zone"] = (X.pu_zone == X.do_zone).astype(int)
    for c in ("pu_zone", "do_zone"):
        X[c] = pd.Categorical(X[c], categories=ZONE_CATS)
    for c in ("pu_b", "do_b"):
        X[c] = pd.Categorical(X[c], categories=BORO_CATS)
    return X


def load_trips(mode, month_filter, max_rows=None):
    d = (spark.table(f"{S}.trips_zoned").filter(F.col("mode") == mode)
         .filter(F.col("dow").between(2, 6) & F.col("hour").isin(HOURS) & month_filter)
         .select("pu_zone", "do_zone", "hour", "dow", "month", "minutes"))
    if max_rows:
        n = d.count()
        if n > max_rows:
            d = d.sample(False, max_rows / n, seed=42)
    return d.toPandas()


def target(minutes):
    return np.log1p(minutes) if LOG_TARGET else minutes


def train_mode(trips):
    """One model per quantile: early stopping on the latest training month, then refit on all training months."""
    X, y = build_features(trips)[FEATURES], target(trips.minutes.to_numpy())
    val = (trips.month == trips.month.max()).to_numpy()
    assert val.any() and (~val).any(), "need at least two training months"
    fitted, best = {}, {}
    for q in QUANTILES:
        m = lgb.LGBMRegressor(alpha=q, **PARAMS)
        m.fit(X[~val], y[~val], eval_X=X[val], eval_y=y[val], eval_metric="quantile",
              callbacks=[lgb.early_stopping(50, verbose=False)])
        best[q] = int(m.best_iteration_ or PARAMS["n_estimators"])
        fitted[q] = lgb.LGBMRegressor(alpha=q, **{**PARAMS, "n_estimators": best[q]}).fit(X, y)
    return fitted, best


def predict_minutes(mode_models, X):
    """Columns are p10, p50, p90 in minutes. Sorted so a lower quantile never ends up above a higher one."""
    p = np.column_stack([mode_models[q].predict(X) for q in QUANTILES])
    if LOG_TARGET:
        p = np.expm1(p)
    return np.clip(np.sort(p, axis=1), 0, None)

In [ ]:
models, best_iters = {}, {}
for mode in MODES:
    t0 = time.time()
    tr = load_trips(mode, F.col("month") < TEST_MONTH, MAX_TRAIN_ROWS)
    models[mode], best_iters[mode] = train_mode(tr)
    imp = pd.Series(models[mode][0.5].booster_.feature_importance("gain"), index=FEATURES)
    print(f"{mode}: {len(tr):,} training trips, best iterations {best_iters[mode]}, {time.time() - t0:.0f}s")
    print("   top features (p50):", imp.nlargest(5).round(0).astype(int).to_dict())

## Test on October and compare with the baseline

In [ ]:
def baseline_tables():
    names = [f"{S}.baseline_zone", f"{S}.baseline_borough"]
    if not all(spark.catalog.tableExists(n) for n in names):
        print("baseline tables not found (run the last cell of 04 first), skipping the comparison")
        return None, None
    return [spark.table(n).toPandas() for n in names]

bz, bb = baseline_tables()


def add_baseline(res, X, mode):
    """Attach the 05 baseline to each test trip: zone pair first, borough pair as the fallback."""
    for c in ("base_p10", "base_p50", "base_p90"):
        res[c] = np.nan
    if bz is None:
        return res
    k = pd.DataFrame({"pu_zone": res.pu_zone.astype(int), "do_zone": res.do_zone.astype(int),
                      "pu_b": X.pu_b.astype(str), "do_b": X.do_b.astype(str)})
    z = bz[bz["mode"] == mode][["pu_zone", "do_zone", "med", "p10", "p90"]]
    b = bb[bb["mode"] == mode][["pu_b", "do_b", "med", "p10", "p90"]]
    k = (k.merge(z, how="left", on=["pu_zone", "do_zone"])
          .merge(b, how="left", on=["pu_b", "do_b"], suffixes=("_z", "_b")))
    for new, c in [("base_p10", "p10"), ("base_p50", "med"), ("base_p90", "p90")]:
        res[new] = k[f"{c}_z"].fillna(k[f"{c}_b"]).to_numpy()
    return res


def score(y, lo, mid, hi):
    return {"MAE_min": np.abs(y - mid).mean(),
            "coverage": ((y >= lo) & (y <= hi)).mean(),
            "width_min": (hi - lo).mean()}

In [ ]:
rows, preds, X_check = [], [], {}
for mode in MODES:
    te = load_trips(mode, F.col("month") == TEST_MONTH).reset_index(drop=True)
    Xf = build_features(te)
    X_check[mode] = Xf[FEATURES].head(1000)
    p = predict_minutes(models[mode], Xf[FEATURES])

    res = te[["pu_zone", "do_zone", "hour", "dow", "minutes"]].copy()
    res.insert(0, "mode", mode)
    res["gbm_p10"], res["gbm_p50"], res["gbm_p90"] = p[:, 0], p[:, 1], p[:, 2]
    res = add_baseline(res, Xf, mode)
    preds.append(res)

    # compare on the same trips: the ones the baseline can cover
    has_base = res.base_p50.notna().any()
    if not has_base:
        print(f"{mode}: no baseline rows found (rerun the last cell of 04), scoring the gbm on all {len(res):,} test trips")
    cmp = res[res.base_p50.notna()] if has_base else res
    rows.append({"mode": mode, "model": "gbm", "test_trips": len(cmp),
                 **score(cmp.minutes, cmp.gbm_p10, cmp.gbm_p50, cmp.gbm_p90)})
    if has_base:
        rows.append({"mode": mode, "model": "baseline", "test_trips": len(cmp),
                     **score(cmp.minutes, cmp.base_p10, cmp.base_p50, cmp.base_p90)})

metrics = pd.DataFrame(rows).round(3)
print(metrics.to_string(index=False))

preds = pd.concat(preds, ignore_index=True)
spark.createDataFrame(metrics).write.mode("overwrite").saveAsTable(f"{S}.gbm_metrics")
spark.createDataFrame(preds).write.mode("overwrite").saveAsTable(f"{S}.gbm_test_predictions")

## Save the models

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
for mode in MODES:
    for q in QUANTILES:
        # write the model text ourselves: LightGBM's save_model needs to seek, and volumes do not allow that
        with open(f"{MODEL_DIR}/{mode}_q{round(q * 100)}.txt", "w") as f:
            f.write(models[mode][q].booster_.model_to_string())

spec = {"features": FEATURES, "categorical": CAT_COLS, "hours": HOURS,
        "quantiles": QUANTILES, "log_target": LOG_TARGET, "test_month": TEST_MONTH,
        "zone_categories": ZONE_CATS, "borough_categories": BORO_CATS,
        "best_iterations": {m: {str(q): n for q, n in d.items()} for m, d in best_iters.items()}}
with open(f"{MODEL_DIR}/model_spec.json", "w") as f:
    json.dump(spec, f, indent=1)

# lookups the app needs to rebuild the same features
zones.to_csv(f"{MODEL_DIR}/zone_features.csv", index=False)
traffic.to_csv(f"{MODEL_DIR}/traffic_features.csv", index=False)
print(sorted(os.listdir(MODEL_DIR)))

In [ ]:
# check: saved models reload and predict the same as the ones in memory
for mode in MODES:
    for q in QUANTILES:
        with open(f"{MODEL_DIR}/{mode}_q{round(q * 100)}.txt") as f:
            b = lgb.Booster(model_str=f.read())
        diff = np.abs(b.predict(X_check[mode]) - models[mode][q].predict(X_check[mode])).max()
        assert diff < 1e-6, (mode, q, diff)
print("saved models reload and give the same predictions")